# Feature engineering — feature engineering and representation comparison

**CSE437 Data Science | Group 15**

feature engineering defines eight derived fields. representation comparison below demonstrates supervised feature selection and centered numeric PCA, compares four representations with a fixed logistic-regression reference, and records the preferred representation. Model-family comparison and tuning are documented in Notebook 04.

The original proposal, target, cohort and frozen folds are preserved. feature engineering reads no targets; representation comparison uses development training labels for selection/reference fitting and development validation labels for comparison. No held-out rows are fitted, transformed or scored.

**Execution provenance:** the five feature engineering code cells retain their previously executed Python outputs. The five appended representation comparison code cells executed sequentially in a new Python process with actual stdout captured. Jupyter/IPython/nbformat remain unavailable; a full fresh Jupyter-kernel run and canonical format validation are still final submission gates.


## Running the analysis

[Open in Google Colab](https://colab.research.google.com/github/faraaz1027-cloud/cse437-hotel-cancellation-15/blob/main/notebooks/03_feature_engineering.ipynb)

Choose **Runtime > Run all**. Setup downloads the project, verifies inputs and installs pinned analysis packages. If it reports **SETUP PAUSED**, choose **Runtime > Restart session**, then **Run all** again; do not delete the runtime. No manual data upload or Drive mount is required. CPU is sufficient.

The source revision is printed during setup. Existing local checkouts are not reset. Download the executed notebook to retain outputs. Recorded outputs below are historical; path/label cleanup and this setup cell do not constitute a new execution. Current Colab execution remains unverified. Development reruns can differ numerically; cached final evaluation is not a new model fit.

In [ ]:
"""Self-contained Colab bootstrap embedded verbatim in the five notebooks.

Uses only the standard library until the analysis dependencies are ready.
Does not modify a user's checkout, install Jupyter, or weaken scientific checks.
"""
import hashlib
import importlib.metadata
import os
from pathlib import Path
import subprocess
import sys


SOURCE_URL = 'https://github.com/faraaz1027-cloud/cse437-hotel-cancellation-15.git'
SOURCE_COMMIT = '77ae899ba79f4f78a4d1c551fae1d08ea42c4765'
ANALYSIS_PACKAGES = {
    'numpy': ('numpy', '2.3.5'),
    'pandas': ('pandas', '2.2.3'),
    'scipy': ('scipy', '1.17.0'),
    'scikit-learn': ('sklearn', '1.8.0'),
    'matplotlib': ('matplotlib', '3.10.8'),
    'seaborn': ('seaborn', '0.13.2'),
    'joblib': ('joblib', '1.5.3'),
}
PROTECTED_FILES = {
    'data/raw/hotel_bookings.csv':
        '7c2ae42a7353905ea136e5c2287f17c92c5435826598bfbb8491c6f0c7b1fc06',
    'data/processed/results/tuning/final_selection.json':
        '68c4072f5c95e3a9f927a8b70a9e96aea8adbd4f7509f4d1c30ba6f7889f3b1b',
    'models/final_logistic_regression.joblib':
        '498112adf28d66f22f84f76101187c7a94eefeda62b7a92b45f1f9152790e097',
}


def in_colab():
    return 'google.colab' in sys.modules or bool(os.environ.get('COLAB_RELEASE_TAG'))


def installed_version(package):
    try:
        return importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        return None


def ensure_analysis_packages():
    # Do not replace Colab's IPython, ipykernel, or notebook server.
    changed = {name for name, (_, version) in ANALYSIS_PACKAGES.items()
               if installed_version(name) != version}
    loaded_before = {name for name, (module, _) in ANALYSIS_PACKAGES.items()
                     if module in sys.modules}
    if changed:
        print('Installing pinned analysis packages. This may take a few minutes.', flush=True)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
                        *[f'{name}=={version}' for name, (_, version)
                          in ANALYSIS_PACKAGES.items()]], check=True)
    for name, (_, version) in ANALYSIS_PACKAGES.items():
        if installed_version(name) != version:
            raise RuntimeError(f'{name} installation did not reach {version}; stop and inspect pip output.')
    stale = changed & loaded_before
    for name, (module, version) in ANALYSIS_PACKAGES.items():
        loaded = sys.modules.get(module)
        if loaded is not None and getattr(loaded, '__version__', version) != version:
            stale.add(name)
    if stale:
        raise RuntimeError(
            'SETUP PAUSED: packages already loaded in memory need a restart: '
            + ', '.join(sorted(stale))
            + '. Choose Runtime > Restart session, then Runtime > Run all. '
              'Do not disconnect/delete the runtime. Installed packages are retained.')


def repository_at_or_above(folder):
    folder = Path(folder).resolve()
    for candidate in (folder, *folder.parents):
        if ((candidate / 'src/eligibility.py').is_file()
                and (candidate / 'data/processed/splits/validation_split_plan.json').is_file()
                and (candidate / 'notebooks').is_dir()):
            return candidate
    return None


def checked_checkout(destination):
    destination = Path(destination)
    if not destination.exists():
        destination.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(['git', '-c', 'core.autocrlf=false', 'clone', '--no-checkout', SOURCE_URL, str(destination)], check=True)
        subprocess.run(['git', '-C', str(destination), 'checkout', '--detach', SOURCE_COMMIT], check=True)
    elif not (destination / '.git').is_dir():
        raise RuntimeError('The setup folder already exists but is not a Git checkout. '
                           'Use a fresh Colab runtime; no existing files were overwritten.')
    head = subprocess.check_output(['git', '-C', str(destination), 'rev-parse', 'HEAD'], text=True).strip()
    origin = subprocess.check_output(['git', '-C', str(destination), 'remote', 'get-url', 'origin'], text=True).strip()
    if head != SOURCE_COMMIT or origin != SOURCE_URL:
        raise RuntimeError('Existing checkout has a different source/version. '
                           'Use a fresh runtime. Setup will not reset or overwrite it.')
    return destination


def verify_inputs(root):
    for relative, expected in PROTECTED_FILES.items():
        path = Path(root) / relative
        if not path.is_file():
            raise FileNotFoundError(f'Missing required project input: {relative}')
        if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            raise RuntimeError(f'Protected input differs: {relative}. No replacement was attempted.')


def prepare_colab():
    if not in_colab():
        # The ordinary local/verification workflow already supplies its environment.
        return None
    if sys.version_info[:2] not in ((3, 12), (3, 13)):
        raise RuntimeError('These package pins need Python 3.12 or 3.13. '
                           'Use a compatible Colab runtime, or the documented local Python 3.12 setup.')
    ensure_analysis_packages()
    root = repository_at_or_above(Path.cwd())
    if root is None:
        root = checked_checkout(Path('/content/cse437-colab') / SOURCE_COMMIT)
        print('Analysis source commit:', SOURCE_COMMIT)
    else:
        print('Using the existing project working directory; no checkout reset or update.')
    verify_inputs(root)
    os.chdir(root)
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))
    print('Setup ready:', root)
    print('CPU runtime is sufficient. No Google Drive mount or manual data upload is needed.')
    print('Development scores may vary; notebook 05 verifies the original cached test results.')
    return root


if __name__ == '__main__':
    prepare_colab()


In [1]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir() and (ROOT.parent / "src").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "data/processed/splits/validation_split_plan.json").is_file(), "Run from the repo root or notebooks directory."
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.feature_engineering import BookingFeatureEngineer, make_feature_preprocessor, DERIVED_COLUMNS
from src.feature_audit import run_feature_audit
from src.preprocessing import LOG_COLUMNS, NUMERIC_COLUMNS, CATEGORICAL_COLUMNS
print("feature engineering: fixed features; training-fold imputation and encoding; test untouched.")


feature engineering: fixed features; training-fold imputation and encoding; test untouched.


## Feature decisions

| Derived field | Formula / interpretation |
| --- | --- |
| `total_nights` | Weekend nights + weekday nights; zero stays retained |
| `total_guests` | Adults + children + babies; unknown components propagate |
| `previous_bookings_total` | Previous cancellations + previous noncanceled bookings |
| `has_booking_history` | 1 if history total > 0, 0 if zero, missing if unknown |
| `previous_cancellation_share` | Cancellations / history total; zero for no history, interpreted together with the history flag |
| `company_code_recorded` | 1 if company code is present, 0 if null; literal code 0 is present |
| `arrival_month_sin` | sin(2π(month−1)/12) |
| `arrival_month_cos` | cos(2π(month−1)/12) |

Month names are replaced with the two fixed calendar coordinates; other source fields are retained for later selection. Company IDs remain excluded. The company flag represents code recording, not verified payment or booking-time availability. Unknown or invalid calendar month names fail explicitly.

Totals and the ratio are formed **before** training-fold imputation. The pipeline logs the three nonnegative totals along with preprocessing's four log fields. Ratios, flags and cyclic coordinates are not logged. Separate median-imputed totals need not equal sums of imputed components. Fixed missing indicators retain children, ADR, total nights, total guests and cancellation-share missingness.


In [2]:
summary, statistics = run_feature_audit(ROOT)
print(json.dumps({k: summary[k] for k in ["development_rows", "retained_source_fields", "derived_fields",
    "fields_before_encoding", "fixed_missing_indicators", "development_zero_night_bookings_retained",
    "development_unknown_total_guests", "development_no_recorded_history", "development_company_code_recorded",
    "test_rows_fitted_or_transformed", "target_values_read", "predictive_models_trained"]}, indent=2))


{
  "development_rows": 95415,
  "retained_source_fields": 24,
  "derived_fields": [
    "total_nights",
    "total_guests",
    "previous_bookings_total",
    "has_booking_history",
    "previous_cancellation_share",
    "company_code_recorded",
    "arrival_month_sin",
    "arrival_month_cos"
  ],
  "fields_before_encoding": 32,
  "fixed_missing_indicators": [
    "children",
    "adr",
    "total_nights",
    "total_guests",
    "previous_cancellation_share"
  ],
  "development_zero_night_bookings_retained": 604,
  "development_unknown_total_guests": 4,
  "development_no_recorded_history": 86585,
  "development_company_code_recorded": 5900,
  "test_rows_fitted_or_transformed": 0,
  "target_values_read": false,
  "predictive_models_trained": 0
}


## Observed development feature values

These are fixed-formula diagnostics, not performance results or fitted full-development transformations. Missing values below precede imputation.


In [3]:
print(statistics.round(4).to_string(index=False))


                    feature   count    mean    std  min    25%  50%   75%  max  missing_before_imputation
               total_nights 95415.0  3.3562 2.5564  0.0  2.000  3.0 4.000 69.0                          0
               total_guests 95411.0  1.9434 0.7259  1.0  2.000  2.0 2.000 55.0                          4
    previous_bookings_total 95415.0  0.2395 1.7702  0.0  0.000  0.0 0.000 67.0                          0
        has_booking_history 95415.0  0.0925 0.2898  0.0  0.000  0.0 0.000  1.0                          0
previous_cancellation_share 95415.0  0.0625 0.2404  0.0  0.000  0.0 0.000  1.0                          0
      company_code_recorded 95415.0  0.0618 0.2409  0.0  0.000  0.0 0.000  1.0                          0
          arrival_month_sin 95415.0 -0.0484 0.7384 -1.0 -0.866  0.0 0.866  1.0                          0
          arrival_month_cos 95415.0 -0.0066 0.6726 -1.0 -0.500 -0.0 0.500  1.0                          0


## Training-fold verification

Each variant is fitted separately on the training prefix. Validation transformation must not change fitted state. Vocabulary, medians and scaling are checked against that training prefix; no test data enter the checks.


In [4]:
fold_table = pd.DataFrame([{
    "fold": f["fold"], "train_rows": f["training"]["rows"], "validation_rows": f["validation"]["rows"],
    "encoded_columns": f["training"]["columns"], "validation_nonfinite": f["validation"]["nonfinite_values"],
    "unseen_month_rows": f["validation_rows_with_unseen_month"],
    "validation_kept_fitted_state": f["validation_did_not_change_fitted_state"]
} for f in summary["folds"]])
print(fold_table.to_string(index=False))
print("\nMonth names absent from training:")
for fold in summary["folds"]:
    print(fold["fold"], fold["validation_months_absent_from_training"])
assert summary["fields_before_encoding"] == 32
assert all(f["scaled_and_unscaled_variants_verified"] for f in summary["folds"])
assert all(f["validation"]["nonfinite_values"] == 0 for f in summary["folds"])
print("\nBoth model-compatible variants passed all three frozen folds.")


 fold  train_rows  validation_rows  encoded_columns  validation_nonfinite  unseen_month_rows  validation_kept_fitted_state
    1       23797            23893              332                     0              23475                          True
    2       47690            23776              421                     0                  0                          True
    3       71466            23949              490                     0                  0                          True

Month names absent from training:
1 ['April', 'February', 'June', 'March', 'May']
2 []
3 []

Both model-compatible variants passed all three frozen folds.


## Synthetic examples for explanation

The following three rows are invented demonstrations of formulas, not source records. No-history and observed zero cancellation share have different history flags. A missing child count leaves the guest total missing until fold-fitted imputation.


In [5]:
row = {c: 1 for c in LOG_COLUMNS + NUMERIC_COLUMNS}
row.update({c: "A" for c in CATEGORICAL_COLUMNS})
row.update(arrival_date_month="January", company=np.nan, agent=1, adults=2, babies=0,
           assigned_room_type="A", booking_changes=0, days_in_waiting_list=0)
examples = pd.DataFrame([row.copy() for _ in range(3)])
examples["children"] = [0, np.nan, 1]
examples["previous_cancellations"] = [0, 0, 2]
examples["previous_bookings_not_canceled"] = [0, 4, 6]
examples["company"] = [np.nan, 0, 42]
examples["arrival_date_month"] = ["December", "January", "June"]
examples["stays_in_weekend_nights"] = [0, 1, 2]
examples["stays_in_week_nights"] = [0, 4, 3]
result = BookingFeatureEngineer().fit_transform(examples)
assert result.previous_cancellation_share.tolist() == [0, 0, .25]
assert result.has_booking_history.tolist() == [0, 1, 1]
assert np.isnan(result.total_guests.iloc[1])
print(result[list(DERIVED_COLUMNS)].round(4).to_string(index=False))


 total_nights  total_guests  previous_bookings_total  has_booking_history  previous_cancellation_share  company_code_recorded  arrival_month_sin  arrival_month_cos
          0.0           2.0                      0.0                  0.0                         0.00                    0.0               -0.5              0.866
          5.0           NaN                      4.0                  1.0                         0.00                    1.0                0.0              1.000
          5.0           3.0                      8.0                  1.0                         0.25                    1.0                0.5             -0.866


## Representation comparison

The candidate schema has 32 fields before encoding. The following representation comparison section fits selection and PCA inside training folds and compares representations. The feature engineering evidence above is preserved as a historical checkpoint; later predictive evaluation does not change those deterministic feature definitions.


## representation comparison — Feature selection and dimensionality reduction

The fixed comparison includes `full`, `selected`, `pca`, and `selected_pca`. Selection excludes training variance ≤1e−12, ranks the remaining encoded features by training-label ANOVA F, and retains the top 75% (rounded upward; ties follow encoded order). F scores are ranking heuristics; p-values and causal interpretations are not used.

PCA uses centered full SVD on only the scaled numeric fields, retaining at least 95% of their training variance. Categories and missing indicators stay sparse and are not mixed into PCA. `selected_pca` applies selection first, then PCA to its retained numeric fields. These choices are written to `comparison_protocol.json` before evaluation.

One fixed logistic-regression reference uses C=1, lbfgs, max_iter=2000, tol=1e−4, class_weight=None, seed=42, and threshold=0.5. All four versions share the three frozen forward folds. Highest unweighted mean cancellation F1 determines the current preference; exact ties favor fewer output columns and then declared mode order. This does not complete model comparison's baseline/two-family comparison or tuning's model tuning.


In [6]:
from pathlib import Path
import sys
import json
import pandas as pd
ROOT=Path.cwd().resolve()
if not (ROOT/"src").is_dir() and (ROOT.parent/"src").is_dir():
    ROOT=ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0,str(ROOT))
from src.representation_audit import run_representation_audit
from src.representation import BookingRepresentation
representations, comparison, fold_results = run_representation_audit(ROOT)
print("Final test rows processed:",representations["test_rows_fitted_transformed_or_scored"])


representation comparison fold 1 full: F1=0.643231, width=332
representation comparison fold 1 selected: F1=0.693691, width=247
representation comparison fold 1 pca: F1=0.692187, width=325
representation comparison fold 1 selected_pca: F1=0.692859, width=242
representation comparison fold 2 full: F1=0.702455, width=421
representation comparison fold 2 selected: F1=0.714117, width=314
representation comparison fold 2 pca: F1=0.662830, width=413
representation comparison fold 2 selected_pca: F1=0.680961, width=308
representation comparison fold 3 full: F1=0.733595, width=490
representation comparison fold 3 selected: F1=0.733020, width=366
representation comparison fold 3 pca: F1=0.728881, width=483
representation comparison fold 3 selected_pca: F1=0.729248, width=360
Final test rows processed: 0


### Development comparison

These scores select a representation using a fixed reference model; they are not untouched-test results. Fold standard deviation is descriptive, not a confidence interval. Reuse of the development folds for choosing a version can make the chosen score optimistic.


In [7]:
print(comparison.round(6).to_string(index=False))
print("\nCurrent preferred mode:",representations["preferred_mode"])
print("Mean F1 difference from full features:",round(representations["mean_f1"]-representations["all_feature_reference_mean_f1"],6))
print("\nPer-fold F1:")
print(fold_results.pivot(index="fold",columns="mode",values="f1").round(6).to_string())


        mode  mean_f1  fold_sd_f1  mean_accuracy  mean_precision  mean_recall  mean_roc_auc  mean_output_columns
        full 0.693094    0.045904       0.751370        0.681025     0.753386      0.876098           414.333333
    selected 0.713609    0.019669       0.809314        0.783364     0.659498      0.882905           309.000000
         pca 0.694633    0.033093       0.805492        0.806468     0.615962      0.882957           407.000000
selected_pca 0.701023    0.025158       0.806410        0.796999     0.630852      0.882036           303.333333

Current preferred mode: selected
Mean F1 difference from full features: 0.020516

Per-fold F1:
mode      full       pca  selected  selected_pca
fold                                            
1     0.643231  0.692187  0.693691      0.692859
2     0.702455  0.662830  0.714117      0.680961
3     0.733595  0.728881  0.733020      0.729248


### Reduction and retained training information

The PCA variance target applies only to the numeric block entering PCA. It is not a claim about retaining 95% of all information or predictive accuracy. Complete component coefficients and input names are exported with the per-fold schemas.

![Numeric PCA training variance](../figures/06_numeric_pca_variance.png)


In [8]:
columns=["fold","mode","encoded_inputs","selected_inputs","numeric_inputs_to_pca",
         "pca_components","pca_retained_numeric_variance","output_columns"]
print(fold_results[columns].round(6).to_string(index=False))
assert fold_results.training_only_state_verified.all()
assert fold_results.loc[fold_results.pca_components.gt(0),"pca_retained_numeric_variance"].ge(.95).all()
print("\nLargest dense numeric training block (bytes):",representations["max_dense_numeric_training_bytes"])


 fold         mode  encoded_inputs  selected_inputs  numeric_inputs_to_pca  pca_components  pca_retained_numeric_variance  output_columns
    1         full             332              332                      0               0                            NaN             332
    1     selected             332              247                      0               0                            NaN             247
    1          pca             332              332                     23              16                       0.958830             325
    1 selected_pca             332              247                     20              15                       0.964056             242
    2         full             421              421                      0               0                            NaN             421
    2     selected             421              314                      0               0                            NaN             314
    2          pca             421

### Retained fields and interpretation

Different training-fold vocabularies and scores produce different retained feature names. The full lists and rankings are saved; a single full-development mask is deliberately not fitted yet. The following top-ranked names describe the third training fold only and are not causal importance estimates.


In [9]:
rankings=pd.read_csv(ROOT/"data/processed/representations/feature_rankings.csv")
print(rankings.query("fold==3").head(12)[["rank","feature","f_score","selected"]].to_string(index=False))
print("\nSelected encoded fields by fold:")
print(rankings.groupby("fold")["selected"].sum().to_string())
schemas=json.loads((ROOT/"data/processed/representations/representation_schemas.json").read_text())
for fold in range(1,4):
    record=schemas[f"fold_{fold}"][representations["preferred_mode"]]
    assert len(record["output_names"])==int(fold_results.loc[(fold_results.fold==fold)&(fold_results["mode"]==representations["preferred_mode"]),"output_columns"].iloc[0])
print("Complete preferred feature lists match measured output widths.")


 rank                              feature      f_score  selected
    1 categorical__deposit_type_Non Refund 26783.391628      True
    2 categorical__deposit_type_No Deposit 25918.206543      True
    3             categorical__country_PRT 13468.750871      True
    4 numeric__previous_cancellation_share 12745.434276      True
    5               log_numeric__lead_time  8739.763169      True
    6  log_numeric__previous_cancellations  8257.270540      True
    7         numeric__has_booking_history  6727.154412      True
    8   numeric__total_of_special_requests  5011.885657      True
    9   categorical__market_segment_Groups  4958.973908      True
   10                 categorical__agent_1  4239.046957      True
   11 numeric__required_car_parking_spaces  3002.815587      True
   12 log_numeric__previous_bookings_total  2422.944264      True

Selected encoded fields by fold:
fold
1    247
2    314
3    366
Complete preferred feature lists match measured output widths.


### Decision and limitations

Selection alone is retained as the current reference-model choice because its mean F1 is highest; the improvement is not uniform across folds. PCA was implemented and evaluated but is not retained in the preferred pipeline because it scored lower. This is a result to report, not a reason to hide the PCA experiment.

Rare-category signal and nonlinear interactions may be missed by univariate ranking. The random-forest ranking may differ from logistic regression, so model comparison must keep a full-feature control. Source timing, repeated records, and later-season shift remain limitations. No held-out result has guided the decision.


In [10]:
preferred=BookingRepresentation(mode=representations["preferred_mode"],percentile=75,variance_target=.95,scale_numeric=True)
assert not hasattr(preferred,"preprocessor_")
assert representations["preferred_mode"]=="selected"
assert representations["reference_model_fits"]==12
assert representations["test_rows_fitted_transformed_or_scored"]==0
print("No final full-development model fitted; model tuning and held-out evaluation remain additional analyses.")


No final full-development model fitted; model tuning and held-out evaluation remain additional analyses.
